# Tutorial 2: PCA + GP Emulator Training

Directly running the physics simulation for every MCMC proposal is
computationally infeasible. Instead, we:

1. **Compress** the high-dimensional simulation output with SVD-PCA
2. **Train** one Gaussian Process (GP) per principal component
3. **Validate** on a held-out set

The trained emulator replaces the simulator during MCMC at negligible cost.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hic_bayes import fit_pca, Emulator
from hic_bayes.plots import plot_variance_explained, plot_emulator_validation

# Load toy data (run scripts/generate_toy_data.py first)
X      = pd.read_csv('../data/toy/design_points.csv').values
Y      = pd.read_csv('../data/toy/simulation.csv').values
X_val  = pd.read_csv('../data/toy/validation_design.csv').values
Y_val  = pd.read_csv('../data/toy/validation_simulation.csv').values

print(f'Training:   X={X.shape}, Y={Y.shape}')
print(f'Validation: X_val={X_val.shape}, Y_val={Y_val.shape}')

In [ ]:
# Fit PCA — choose n_components to capture >= 95% of variance
pca = fit_pca(Y, n_components=5)

print('Variance explained by each PC:')
for i, v in enumerate(pca.variance_explained):
    print(f'  PC{i+1}: {v:.2%}  (cumulative: {pca.cumulative_variance[i]:.2%})')

fig = plot_variance_explained(pca.variance_explained, max_pcs=10)
plt.show()

In [ ]:
# Project training data into PC space
Z_train = pca.transform(Y)
print(f'PC score matrix shape: {Z_train.shape}')

# Parameter ranges for GP kernel initialisation
param_mins = X.min(axis=0)
param_maxs = X.max(axis=0)
param_ranges = np.column_stack([param_mins, param_maxs])

# Train emulator (one GP per PC)
emulator = Emulator(n_restarts=4)
emulator.fit(X, Z_train, param_ranges, n_jobs=-1)
print('Emulator training complete.')

In [ ]:
# Predict on validation set
Z_val_pred, Z_val_std = emulator.predict(X_val, return_std=True)

# Reconstruct observables from PC scores
Y_val_pred = pca.inverse_transform(Z_val_pred)
Y_val_std  = pca.inverse_transform(Z_val_std)  # approximate propagation

# Compute R² per observable
ss_res = np.sum((Y_val - Y_val_pred)**2, axis=0)
ss_tot = np.sum((Y_val - Y_val.mean(axis=0))**2, axis=0)
r2 = 1.0 - ss_res / ss_tot
print(f'Mean R² over all observables: {r2.mean():.4f}')
print(f'Min R²: {r2.min():.4f}  |  Max R²: {r2.max():.4f}')

fig = plot_emulator_validation(Y_val, Y_val_pred, n_cols=5)
plt.suptitle('Emulator validation: predicted vs true', y=1.01)
plt.show()

## Tips

- If R² is low for some observables, increase `n_components` in the PCA step.
- Low-variance observables (dominated by numerical noise) can be excluded from the likelihood.
- The `n_restarts` parameter controls how well each GP's hyperparameters are optimised — increase for better accuracy at the cost of training time.